# SOFR Curve Screener — Pipeline

Infrastructure for screening curve / fly dislocations.

| Object | Role |
|---|---|
| `data_raw` | 3h SOFR swap tenors from Excel/CSV |
| `full_data` | All curves + flies (`2s5s`, `10s30s`, `5s7s10s`, …) |
| `predicted_data` | Isolated targets (start: `30s35s`) |
| `RollingResult` | Walk-forward OLS preds + accuracy |

Modules: `curve_config` · `data_loader` · `structures` · `rolling_model`

In [ ]:
from pathlib import Path
import logging

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from curve_config import (
    CURVES, FLIES, PREDICTED, REGRESSIONS, TRAIN_WINDOWS,
    DATA_PATH, TRAIN_SIZE, TEST_SIZE, STEP_SIZE,
)
from data_loader import load_raw, generate_synthetic_sofr
from structures import build_datasets
from rolling_model import (
    run_regressions, predictions_panel,
    instance_predictions, plot_residual_evolution,
)

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)

print("curves :", [c[0] for c in CURVES])
print("flies  :", [f[0] for f in FLIES])
print("targets:", PREDICTED)
print("models :", [r.name for r in REGRESSIONS])
print(f"train windows to compare: {TRAIN_WINDOWS}")
print(f"walk-forward: test={TEST_SIZE}  step={STEP_SIZE}")


## 1. Load `data_raw`

Point `DATA_PATH` in `curve_config.py` (or override below) at your Excel/CSV.
If the file is missing, we fall back to synthetic SOFR so the rest of the notebook still runs.

In [ ]:
path = Path(DATA_PATH)  # e.g. Path("/datazone/sofr_3h.xlsx")

if path.exists():
    data_raw = load_raw(path)
    print(f"Loaded real file: {path.resolve()}")
else:
    data_raw = generate_synthetic_sofr(n_bars=400)
    print(f"File not found ({path}) — using synthetic SOFR panel")

data_raw.head()

## 2. Build `full_data` and `predicted_data`

- **Curve** `XsYs` = `(Y − X) * 100` (bp)
- **Fly** `XsYsZs` = `(2Y − X − Z) * 100` (bp)

Edit `CURVES` / `FLIES` / `PREDICTED` in `curve_config.py` to expand coverage.

In [ ]:
cleaned_raw, full_data, predicted_data = build_datasets(data_raw)

print("cleaned tenors:", list(cleaned_raw.columns))
print("full_data cols:", list(full_data.columns))
print("predicted_data:", list(predicted_data.columns))

display(full_data.tail(3))
display(predicted_data.tail(3))

## 3. Tune train window + walk-forward predictions

Compare `TRAIN_WINDOWS` on OOS RMSE, keep the best, then:
1. plot residual evolution
2. emit one prediction per OOS timestamp


In [ ]:
results = run_regressions(full_data, predicted_data, tune_window=True)

spec = REGRESSIONS[0]
res = results[spec.name]
print(f"Chosen: {spec.name}  train_size={int(res.overall['train_size'])}")
print("Overall OOS:", {k: round(v, 5) if isinstance(v, float) else v for k, v in res.overall.items()})


In [ ]:
# Main output: prediction at each OOS instance
preds = instance_predictions(res)
preds


In [ ]:
chart = plot_residual_evolution(res, outfile="residual_evolution.png")
print("saved:", chart)

# also show inline
fig, ax = plt.subplots(figsize=(12, 4))
ax.axhline(0, color="black", lw=0.8)
ax.plot(preds.index, preds["residual"], color="C3", lw=1.0)
ax.set_title(f"Residual evolution — {spec.name}")
ax.set_ylabel("residual (actual − pred)")
ax.set_xlabel("time")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 4. Wide prediction panel (ready to expand)

As you add `RegressionSpec`s in `curve_config.REGRESSIONS`, this panel grows column-wise.

In [ ]:
pred_panel = predictions_panel(results)
pred_panel.tail()

## 5. How to expand

1. **More structures** — append to `CURVES` / `FLIES` in `curve_config.py`  
2. **More targets** — add names to `PREDICTED` (must already be a curve or fly)  
3. **More models** — append `RegressionSpec(target=..., features=(...))` to `REGRESSIONS`  
4. Re-run cells 2–4 — no other code changes needed